# 01 - Ingest & Clean

## 1. Parse rows with `csv` and save rejects

In [1]:
import csv
import json

def parse_row(row):
    row = dict(row)                               
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])        
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"]) 

# --- Step 2: read and sort rows ---
accepted = []
rejected = []

with open("data/appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    for row in reader:
        try:
            accepted.append(parse_row(row))           
        except ValueError:                        
            rejected.append(row)

# --- Step 3: report ---
print("Total rows:", len(accepted) + len(rejected))
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

# --- Step 4: save rejects ---
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020
Accepted: 988
Rejected: 32


## 2. Load with pandas and inspect

In [2]:
import pandas as pd

df = pd.read_csv("data/appointments.csv")

# --- Inspect ---
print("Shape:", df.shape)                       
df.info()
print(df.isnull().sum())                      
print("Duplicates:", df.duplicated().sum())       
print("Raw channel spellings:", df["booking_channel"].nunique())

Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 170.6 KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
a

## 3. Clean: duplicates, channels, dates, missing

In [3]:
# --- (a) Drop exact duplicates ---
df = df.drop_duplicates()                              
print("After (a):", len(df))

# --- (b) Clean booking_channel ---
df["booking_channel"] = df["booking_channel"].str.strip().str.title()   
print("Clean channel spellings:", df["booking_channel"].nunique())
print(df["booking_channel"].unique())
print("After (b):", len(df))

# --- (c) Convert dates ---
df["appt_date"] = pd.to_datetime(df["appt_date"], errors='coerce') 
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After (c):", len(df))

# --- (d) Drop rows missing appt_date or status ---
df = df.dropna(subset=['appt_date', 'status'])    
print("After (d):", len(df))

After (a): 1000
Clean channel spellings: 3
<ArrowStringArray>
['Phone', 'Walk-In', 'App']
Length: 3, dtype: str
After (b): 1000
After (c): 1000
After (d): 990


## 4. Ages, medians, outliers, derived columns

In [4]:
import numpy as np

# --- (a) Impossible ages become missing ---
bad_age = (df["age"] < 1) | (df["age"] > 100)         
print("Invalid ages:", bad_age.sum())                 
df.loc[bad_age, "age"] = np.nan
print("Missing ages now:", df["age"].isnull().sum())

# --- (b) Fill gaps with the median ---
age_median = df["age"].median()                         
dist_median = df["distance_km"].median()
print("Median age:", age_median, "| Median distance:", dist_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)  
df["distance_km"] = df["distance_km"].fillna(dist_median)

# --- (c) Cap outliers with IQR fences ---
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)                       
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1                                 
    lower = q1 - 1.5 * iqr
    upper = q3  + 1.5 * iqr                                       
    n_capped = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower={lower}, upper={upper}, capped={n_capped}")
    df[col] = np.clip(df[col], lower, upper)           
# --- (d) Round ---
df["distance_km"] = df["distance_km"].round(1)         
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Max fee now:", df["fee_inr"].max())

# --- (e) Derived columns ---
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days      
df["no_show"] = (df["status"] == "No-Show").astype(int)            
print("Status values:", df["status"].unique())
print("Lead days range:", df["lead_days"].min(), "to", df["lead_days"].max())

Invalid ages: 5
Missing ages now: 17
Median age: 48.0 | Median distance: 8.25
distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower=-7.9375, upper=25.5625, capped=26
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower=68.75, upper=1218.75, capped=3
Max fee now: 1219
Status values: <ArrowStringArray>
['No-Show', 'Attended']
Length: 2, dtype: str
Lead days range: 1 to 45


## 5. Clinics + weather API

In [5]:
clinics = pd.read_csv("data/clinics.csv")

import requests

URL = "https://api.open-meteo.com/v1/forecast"

def fetch_temp(lat, lon):
    """Returns (temperature, None) on success, or (None, reason) on failure."""
    params = {"latitude": lat, "longitude": lon,
              "hourly": "temperature_2m", "forecast_days": 1}
    response = requests.get(URL, params=params, timeout=10)     
    if response.status_code != 200:                               
        return None, f"status {response.status_code}"
    data = response.json()                                       
    if "hourly" not in data:
        return None, "no hourly data"
    return data["hourly"]["temperature_2m"][0], None                          

temps = []
for _, row in clinics.iterrows():                 
    temp, reason = None, None
    for attempt in range(2):                    
        try:
            temp, reason = fetch_temp(row["latitude"], row["longitude"])
            if temp is not None:
                break                            
        except requests.exceptions.RequestException as e:   
            reason = str(e)
    if temp is None:
        print("FAILED:", row["clinic_name"], "-", reason)
    else:
        print(row["clinic_name"], "->", temp, "°C")
    temps.append(temp)

clinics["temp_c"] = temps                       
clinics.to_csv("clinics_enriched.csv", index=False)   
print(clinics)

Kothrud Family Clinic -> 17.9 °C
Hadapsar Care Centre -> 18.8 °C
Sitabuldi Health Point -> 20.1 °C
Malviya Nagar Clinic -> 19.6 °C
Vaishali Nagar Clinic -> 19.6 °C
Edappally Medical Centre -> 26.0 °C
Vijay Nagar Clinic -> 21.7 °C
Gomti Nagar Care Point -> 19.5 °C
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.9  


## 6. Export clean appointments

In [6]:
# --- Prepare the clean table ---
cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age",
        "gender", "distance_km", "fee_inr", "booking_channel", "visit_type",
        "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"]

out = df.sort_values("appointment_id")[cols].copy()                      
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")    
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")

# --- Checks before export ---
print("appointment_id unique:", out["appointment_id"].is_unique)
print("Missing values in 16 cols:", out.isnull().sum().sum())  
print("Shape:", out.shape)

out.to_csv("appointments_clean.csv", index=False)

appointment_id unique: True
Missing values in 16 cols: 0
Shape: (990, 16)


## 7. Write `load_data.sql`

In [7]:
# --- Write load_data.sql ---
count = 0
with open("load_data.sql", "w") as f:

    for _, c in clinics.iterrows():
        temp = "NULL" if pd.isna(c["temp_c"]) else c["temp_c"]
        f.write(f"INSERT INTO clinics VALUES ('{c['clinic_id']}','{c['clinic_name']}',"
                f"'{c['city']}','{c['tier']}',{c['latitude']},{c['longitude']},{temp});\n")   
        count += 1

    # then appointments
    for _, r in out.iterrows():
        f.write(f"INSERT INTO appointments VALUES ("
                f"'{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
                f"'{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',"
                f"{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}',"
                f"'{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']}," 
                f"'{r['status']}',{r['lead_days']},{r['no_show']});\n")              
        count += 1

print("INSERT statements written:", count)

INSERT statements written: 998
